# Matplotlib — Figure Architecture and Custom Plots

## Introduction

Matplotlib is the foundation of Python visualization. Every Pandas plot and Seaborn plot is ultimately a Matplotlib figure. Understanding Matplotlib's object model — figures, axes, artists — gives you full control over every element of a chart.

## Objectives

You will be able to:

* Explain the Figure/Axes/Artist object hierarchy
* Create multi-panel figures with `plt.subplots()`
* Customize axes: ticks, labels, limits, spines, gridlines
* Add text, annotations, and reference lines
* Use different plot types: scatter, bar, hist, errorbar, fill_between
* Apply and customize figure styles

---

## The Object Model

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

# Figure is the canvas; Axes is one plot panel
fig, ax = plt.subplots(figsize=(8, 4))

x = np.linspace(0, 2 * np.pi, 100)
ax.plot(x, np.sin(x), label='sin(x)')
ax.plot(x, np.cos(x), label='cos(x)', linestyle='--')

ax.set_title('Sine and Cosine')
ax.set_xlabel('Angle (radians)')
ax.set_ylabel('Value')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# The hierarchy:
#   Figure
#   └── Axes (one or more)
#       ├── Title, xlabel, ylabel
#       ├── XAxis, YAxis
#       │   ├── ticks
#       │   └── tick labels
#       ├── Spines (top, bottom, left, right borders)
#       └── Artists (lines, patches, texts)

print(f"Figure type: {type(fig)}")
print(f"Axes type: {type(ax)}")
print(f"Lines on axes: {len(ax.lines)}")

---

## Multi-Panel Figures

In [ ]:
np.random.seed(42)
data = {
    'scores': np.random.normal(75, 12, 200),
    'study_hrs': np.random.normal(6, 2, 200).clip(0, 12),
}
data['scores'] = (data['study_hrs'] * 5 + np.random.normal(40, 8, 200)).clip(0, 100)
df = pd.DataFrame(data)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# [0,0] Histogram
axes[0, 0].hist(df['scores'], bins=25, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Score Distribution')
axes[0, 0].set_xlabel('Score')
axes[0, 0].set_ylabel('Frequency')

# [0,1] Scatter
axes[0, 1].scatter(df['study_hrs'], df['scores'], alpha=0.4, color='coral', s=20)
axes[0, 1].set_title('Study Hours vs Score')
axes[0, 1].set_xlabel('Study Hours')
axes[0, 1].set_ylabel('Score')

# [1,0] Box plot
grade_bins = pd.cut(df['scores'], bins=[0, 60, 70, 80, 90, 100],
                    labels=['F', 'D', 'C', 'B', 'A'])
counts = grade_bins.value_counts().sort_index()
axes[1, 0].bar(counts.index, counts.values, color='mediumseagreen')
axes[1, 0].set_title('Grade Distribution')
axes[1, 0].set_xlabel('Grade')
axes[1, 0].set_ylabel('Count')

# [1,1] Line: cumulative % passing
thresholds = range(40, 101)
pct_passing = [(df['scores'] >= t).mean() * 100 for t in thresholds]
axes[1, 1].plot(thresholds, pct_passing, color='purple', linewidth=2)
axes[1, 1].axhline(50, color='gray', linestyle='--', linewidth=1, label='50% pass rate')
axes[1, 1].set_title('% Passing at Each Threshold')
axes[1, 1].set_xlabel('Minimum Score')
axes[1, 1].set_ylabel('% Passing')
axes[1, 1].legend()

plt.suptitle('Student Performance Dashboard', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## Axis Customization

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

x = np.linspace(0, 10, 100)
ax.plot(x, x**2, color='navy', linewidth=2)

# Limits
ax.set_xlim(0, 10)
ax.set_ylim(0, 110)

# Custom ticks
ax.set_xticks([0, 2, 4, 6, 8, 10])
ax.set_xticklabels(['0', '2 sec', '4 sec', '6 sec', '8 sec', '10 sec'])

# Reference lines
ax.axhline(50, color='red', linestyle=':', alpha=0.7, label='y=50')
ax.axvline(7.07, color='green', linestyle=':', alpha=0.7, label='x=√50')

# Remove top/right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Annotations
ax.annotate(
    'x²=50 at x≈7.07',
    xy=(7.07, 50), xytext=(3, 80),
    arrowprops=dict(arrowstyle='->', color='gray'),
    fontsize=10, color='gray'
)

ax.set_title('y = x²  with Reference Lines', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
ax.grid(True, alpha=0.2, linestyle='--')

plt.tight_layout()
plt.show()

---

## Useful Plot Types

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Error bars — show uncertainty
methods = ['A', 'B', 'C', 'D']
means = [72, 85, 91, 78]
errors = [5, 3, 4, 8]
axes[0].bar(methods, means, yerr=errors, capsize=6, color='steelblue',
            error_kw={'linewidth': 2})
axes[0].set_title('Method Comparison with Error Bars')
axes[0].set_ylabel('Accuracy (%)')

# fill_between — show confidence interval
x = np.linspace(0, 5, 100)
y = np.sin(x)
uncertainty = 0.3 * np.abs(np.sin(0.5 * x))
axes[1].plot(x, y, color='navy', linewidth=2, label='Mean')
axes[1].fill_between(x, y - uncertainty, y + uncertainty,
                      alpha=0.2, color='navy', label='±1 std')
axes[1].set_title('Confidence Band with fill_between')
axes[1].legend()

# Step plot — useful for histograms and cumulative distributions
x_step = [0, 1, 2, 3, 4, 5]
y_step = [0, 0.2, 0.5, 0.75, 0.9, 1.0]
axes[2].step(x_step, y_step, where='post', color='darkgreen', linewidth=2)
axes[2].set_title('CDF with step()')
axes[2].set_xlabel('Value')
axes[2].set_ylabel('Cumulative Probability')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Styles and Saving

In [ ]:
# Available styles
print([s for s in plt.style.available if 'ggplot' in s or 'seaborn' in s or 'fivethirty' in s])

In [ ]:
# Apply a style
with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(np.random.cumsum(np.random.randn(100)), color='steelblue')
    ax.set_title('Random Walk — seaborn-whitegrid style')
    plt.tight_layout()
    plt.show()

# Saving (use plt.savefig BEFORE plt.show)
# fig.savefig('chart.png', dpi=150, bbox_inches='tight')
# fig.savefig('chart.pdf')  # vector format for publication

---

## Practice

In [ ]:
# Create a 1x2 figure:
# Left: Histogram of 500 random normal values, with a red dashed line at the mean
# Right: Scatter plot of x vs y where y = x^2 + noise, with a smooth polynomial fit line
np.random.seed(42)
normal_data = np.random.normal(100, 15, 500)
x_scat = np.linspace(-3, 3, 100)
y_scat = x_scat**2 + np.random.normal(0, 0.5, 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Your code here
plt.show()

## Summary

| Concept | Key functions |
|---------|---------------|
| Create figure | `fig, ax = plt.subplots(nrows, ncols, figsize=)` |
| Line/scatter | `ax.plot()`, `ax.scatter()` |
| Bar/hist | `ax.bar()`, `ax.hist()` |
| Labels | `ax.set_title()`, `ax.set_xlabel()`, `ax.set_ylabel()` |
| Limits | `ax.set_xlim()`, `ax.set_ylim()` |
| Reference lines | `ax.axhline()`, `ax.axvline()` |
| Annotation | `ax.annotate(text, xy, xytext, arrowprops=)` |
| Spines | `ax.spines['top'].set_visible(False)` |
| Save | `fig.savefig('file.png', dpi=150, bbox_inches='tight')` |

Next: Seaborn — statistical visualizations with intelligent defaults.